# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import xgboost
from xgboost import XGBRegressor


In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2


# 1. Preprocessing

In [4]:
# AUTOREGRESSIVE (LAG) FEATURES

lags = sorted(set(
    list(range(1, 7)) + [12, 18]
    + list(range(24, 27)) + [36, 48]
    + [24*i for i in range(3,7)]
    + [24*7*i for i in range(1,5)]
))
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]

In [5]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    intDateTimeFeatures
    + hourFourierFeatures
    + dayFourierFeatures
    + hourDummyFeatures
    + dayDummyFeatures
    + monthDummyFeatures
)


In [6]:
# ENERGY FEATURES

energyFeatures = [
    "Adjusted net generation",
    "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [7]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + [f"HourlyWindDirection_Flag_{d}" for d in directions]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [8]:
# Define other convenient feature variables.

target = "Adjusted demand"
allFeatures = (
    ['t', target]
    + lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
someFeatures = (
    ['t', target]
    + lagFeatures[:1] 
    + hourFourierFeatures[:2] + dayFourierFeatures[:2] + dayDummyFeatures[:1]
    + energyFeatures[:2] 
    + weatherFeatures[:1]
)


### 1.3 Data Splits

In [9]:
# reproducibility
np.random.seed(0)

# --- paths ---
data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

DFtrain = pd.read_pickle(train_dir / "DFtrain.pkl")
DFval   = pd.read_pickle(val_dir   / "DFval.pkl")

# --- define target and features (ADJUST target name to yours) ---
TARGET_COL   = "Adjusted demand"   # <--- change to your target column
TIME_COL     = "t"        # if you have a time column
FEATURE_COLS = someFeatures[2:]

print("n_train:", len(DFtrain), "n_val:", len(DFval))
print("n_features:", len(FEATURE_COLS))

n_train: 59658 n_val: 12784
n_features: 9


In [10]:
# --- XGBoost hyperparameter tuning on train/val ---

X_train_XGB = DFtrain[FEATURE_COLS].to_numpy()
y_train_XGB = DFtrain[TARGET_COL].to_numpy()
X_val_XGB   = DFval[FEATURE_COLS].to_numpy()
y_val_XGB   = DFval[TARGET_COL].to_numpy()

def eval_xgb(params, X_train, y_train, X_val, y_val):
    """Fit XGBRegressor with given params and return validation RMSE."""
    model = XGBRegressor(
        objective="reg:squarederror",
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=0,
        **params,
    )
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_val_pred)
    return rmse, model

max_depth_grid    = [3, 4, 6]
n_estimators_grid = [200, 400]
learning_rate_grid = [0.05, 0.1]

best_params = None
best_rmse   = np.inf
best_XGB    = None

for max_depth in max_depth_grid:
    for n_estimators in n_estimators_grid:
        for learning_rate in learning_rate_grid:
            params = dict(
                max_depth=max_depth,
                n_estimators=n_estimators,
                learning_rate=learning_rate,
            )
            rmse, model = eval_xgb(params, X_train_XGB, y_train_XGB, X_val_XGB, y_val_XGB)
            print(
                f"XGB depth={max_depth}, trees={n_estimators}, "
                f"lr={learning_rate:.2f}, --- val RMSE: {rmse:.2f}"
            )
            if rmse < best_rmse:
                best_rmse   = rmse
                best_params = params
                best_XGB    = model

print(f"\nBest XGB params={best_params}  (val RMSE={best_rmse:.2f})")


XGB depth=3, trees=200, lr=0.05, --- val RMSE: 386.08
XGB depth=3, trees=200, lr=0.10, --- val RMSE: 353.44
XGB depth=3, trees=400, lr=0.05, --- val RMSE: 353.94
XGB depth=3, trees=400, lr=0.10, --- val RMSE: 341.29
XGB depth=4, trees=200, lr=0.05, --- val RMSE: 346.77
XGB depth=4, trees=200, lr=0.10, --- val RMSE: 337.03
XGB depth=4, trees=400, lr=0.05, --- val RMSE: 331.94
XGB depth=4, trees=400, lr=0.10, --- val RMSE: 329.40
XGB depth=6, trees=200, lr=0.05, --- val RMSE: 331.97
XGB depth=6, trees=200, lr=0.10, --- val RMSE: 334.95
XGB depth=6, trees=400, lr=0.05, --- val RMSE: 327.26
XGB depth=6, trees=400, lr=0.10, --- val RMSE: 333.68

Best XGB params={'max_depth': 6, 'n_estimators': 400, 'learning_rate': 0.05}  (val RMSE=327.26)
